In [1]:
import json
import csv
import os
from datetime import datetime
from confluent_kafka import Consumer, KafkaError, TopicPartition

In [2]:
# kafka configuration
consumer_config = {
    'bootstrap.servers': 'kafka2:9093,kafka1:9092',  # Endereço do(s) broker(s) Kafka
    'group.id': 'consumo-impacta',        # Identificador do grupo de consumidores
    'auto.offset.reset': 'earliest',         # Lê todas as mensagens disponíveis no tópico
    'client.id': 'consumidor_do_kafka'            # nome do client conectado
}
consumer = Consumer(consumer_config)
topic = f'impacta'  # Substitua pelo nome do seu tópico Kafka
partition = 0
offset = 0  # colocando 0, vmaos consumir sempre desde o inicio

# Atribua a partição e o offset ao consumidor
consumer.assign([TopicPartition(topic, partition, offset)])
# consumer.subscribe([topic])

In [ ]:
# Função para salvar mensagens em CSV
def save_to_csv(messages, filename):
    if not messages:
        return
        
    try:
        messages = [json.loads(message.decode('utf-8')) for message in messages]
    except Exception as e:
        print(f"Erro ao decodificar mensagens: {e}")
        return
        
    keys = messages[0].keys()
    output_dir = 'data'
    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)
    
    with open(filepath, 'w', newline='', encoding='utf-8', buffering=1024*8) as output_file:
        dict_writer = csv.DictWriter(output_file, fieldnames=keys)
        dict_writer.writeheader()
        dict_writer.writerows(messages)

In [4]:
# Consumir mensagens e salvar em CSV a cada minuto
messages = []
start_time = datetime.now()

In [ ]:
contador = 0
while contador <= 10:
    msg = consumer.poll(1)
    if msg is None:
        contador += 1
        continue
        
    if msg.error():
        if msg.error().code() == KafkaError._PARTITION_EOF:
            pass # fim da partição, não precisa printar
        else:
            print(f'Error no Consumidor: {msg.error()}')
    else:
        data = msg.value()
        messages.append(data)
        current_time = datetime.now()
        
        if (current_time - start_time).seconds >= 60:
            filename = f"messages_{start_time.strftime('%Y%m%d_%H%M')}.csv"
            save_to_csv(messages, filename)
            print(f"Saved {len(messages)} messages to data/{filename}")
            messages = []
            start_time = current_time
            contador = 0
consumer.close()

0
Recebido: b'{"user": "user2", "action": "comment", "message": "This is a like by user2", "timestamp": "2025-11-19T14:53:53.845075"}'
0
Recebido: b'{"user": "user3", "action": "like", "message": "This is a like by user2", "timestamp": "2025-11-19T14:53:54.855437"}'
0
Recebido: b'{"user": "user1", "action": "comment", "message": "This is a like by user3", "timestamp": "2025-11-19T14:53:55.858570"}'
0
Recebido: b'{"user": "user1", "action": "like", "message": "This is a like by user3", "timestamp": "2025-11-19T14:53:56.859358"}'
0
Recebido: b'{"user": "user2", "action": "comment", "message": "This is a post by user3", "timestamp": "2025-11-19T14:53:57.860140"}'
0
1
Recebido: b'{"user": "user1", "action": "comment", "message": "This is a comment by user1", "timestamp": "2025-11-19T14:53:58.868459"}'
0
1
Recebido: b'{"user": "user2", "action": "post", "message": "This is a post by user1", "timestamp": "2025-11-19T14:53:59.872517"}'
0
1
Recebido: b'{"user": "user3", "action": "post", "mess